In [41]:
import glob
import math
import torch
import torchvision
import torch.nn as nn
from torchinfo import summary
from torchvision.models import vgg19

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
HR_train_paths = sorted(glob.glob("../data/DIV2K_train_HR/*.png"))
X2_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X2/*.png"))
X4_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X4/*.png"))
X8_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X8/*.png"))
X16_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X16/*.png"))
X32_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X32/*.png"))
X64_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X64/*.png"))

HR_valid_paths = sorted(glob.glob("../data/DIV2K_valid_HR/*.png"))
X2_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X2/*.png"))
X4_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X4/*.png"))
X8_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X8/*.png"))
X16_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X16/*.png"))
X32_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X32/*.png"))
X64_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X64/*.png"))

## Generator - SRResNet

In [55]:
class ResBlockSRRN(nn.Module):
        def __init__(self):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv2d(64, 64, 3, stride=1, padding='same'),
                nn.BatchNorm2d(64),
                nn.PReLU(),
                nn.Conv2d(64, 64, 3, stride=1, padding='same'),
                nn.BatchNorm2d(64)
            )

        def forward(self, x):
            return x + self.block(x)

class SRResNet(nn.Module):
    def __init__(self, n: int):
        """
        Args:
            n: scaling factor
        """
        super().__init__()
        
        self.expand = nn.Sequential(
            nn.Conv2d(3, 64, 9, stride=1, padding='same'),
            nn.PReLU()
        )

        self.residual_blocks = nn.Sequential()
        for _ in range(16):
            self.residual_blocks.append(ResBlockSRRN())

        self.residual_blocks.append(nn.Conv2d(64, 64, 3, stride=1, padding='same'))
        self.residual_blocks.append(nn.BatchNorm2d(64))

        self.upscaling_head = nn.Sequential()
        for _ in range(int(math.log2(n))):
            self.upscaling_head.append(nn.Conv2d(64, 256, 3, stride=1, padding='same'))
            self.upscaling_head.append(nn.PixelShuffle(2))
            self.upscaling_head.append(nn.PReLU())
            
        self.upscaling_head.append(nn.Conv2d(64, 3, 9, stride=1, padding='same'))

    def forward(self, x):
        x = self.expand(x)
        return self.upscaling_head(self.residual_blocks(x) + x)

In [56]:
summary(SRResNet(8), input_size=(16, 3, 48, 48))

Layer (type:depth-idx)                   Output Shape              Param #
SRResNet                                 [16, 3, 384, 384]         --
├─Sequential: 1-1                        [16, 64, 48, 48]          --
│    └─Conv2d: 2-1                       [16, 64, 48, 48]          15,616
│    └─PReLU: 2-2                        [16, 64, 48, 48]          1
├─Sequential: 1-2                        [16, 64, 48, 48]          --
│    └─ResBlockSRRN: 2-3                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-1              [16, 64, 48, 48]          74,113
│    └─ResBlockSRRN: 2-4                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-2              [16, 64, 48, 48]          74,113
│    └─ResBlockSRRN: 2-5                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-3              [16, 64, 48, 48]          74,113
│    └─ResBlockSRRN: 2-6                 [16, 64, 48, 48]          --
│    │    └─Sequential: 3-4              [16, 64, 48, 48]          74,

## Discriminator

In [52]:
class ConvBlock(nn.Module):
    def __init__(self, ni: int, nf: int, ks: int, stride: int):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Conv2d(ni, nf, ks, stride=stride, padding=1),
            nn.BatchNorm2d(nf),
            nn.LeakyReLU(negative_slope=0.2)
        )

    def forward(self, x):
        return self.block(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.expand = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1),
            nn.LeakyReLU(negative_slope=0.2)
        )

        self.body = nn.Sequential(
            ConvBlock(64, 64, 3, 2),
            ConvBlock(64, 128, 3, 1),
            ConvBlock(128, 128, 3, 2),
            ConvBlock(128, 256, 3, 1),
            ConvBlock(256, 256, 3, 2),
            ConvBlock(256, 512, 3, 1),
            ConvBlock(512, 512, 3, 2)
        )

        self.avgpool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        self.head = nn.Sequential(
            nn.Linear(512, 1024),
            nn.LeakyReLU(negative_slope=0.2),
            nn.Linear(1024, 1),
            nn.Sigmoid()
        )

        self.model = nn.Sequential(
            self.expand,
            self.body,
            self.avgpool,
            self.head   
        )

    def forward(self, x):
        return self.model(x)

In [53]:
summary(Discriminator(), input_size=(16, 3, 384, 384))

Layer (type:depth-idx)                        Output Shape              Param #
Discriminator                                 [16, 1]                   --
├─Sequential: 1-1                             [16, 1]                   --
│    └─Sequential: 2-1                        [16, 64, 384, 384]        --
│    │    └─Conv2d: 3-1                       [16, 64, 384, 384]        1,792
│    │    └─LeakyReLU: 3-2                    [16, 64, 384, 384]        --
│    └─Sequential: 2-2                        [16, 512, 24, 24]         --
│    │    └─ConvBlock: 3-3                    [16, 64, 192, 192]        37,056
│    │    └─ConvBlock: 3-4                    [16, 128, 192, 192]       74,112
│    │    └─ConvBlock: 3-5                    [16, 128, 96, 96]         147,840
│    │    └─ConvBlock: 3-6                    [16, 256, 96, 96]         295,680
│    │    └─ConvBlock: 3-7                    [16, 256, 48, 48]         590,592
│    │    └─ConvBlock: 3-8                    [16, 512, 48, 48]      

## VGG - Content loss function

In [45]:
vgg = vgg19(weights=torchvision.models.VGG19_Weights.DEFAULT)

In [43]:
vgg

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [51]:
vgg54 = vgg.features[:36]